<a href="https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/codex/fix-madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_load_environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Load a CityLearn v3 MADRL Environment

Este notebook sigue la estructura de `examples/load_environment.ipynb`, pero carga el entorno del proyecto **CityLearn v3 MADRL**. El simulador base sigue siendo CityLearn v2; la capa v3 agrega Dec-POMDP, CTDE, tres ejes `E1/E2/E3`, 17 edificios + EV, recompensa `CityLearnV3MADRLRewardFunction` y perfiles por algoritmo HAPPO, MASAC, MATD3 y MAAC.

En local, use el entorno Python del proyecto. En Google Colab, clone la rama `citylearn-v3-madrl` e instale el repositorio en modo editable. La instalacion queda desactivada por defecto para evitar cambios accidentales.

In [ ]:
# Controla si se ejecuta la instalacion en Google Colab.
RUN_COLAB_INSTALL = False

# Importa utilidades del sistema operativo.
import os

# Importa rutas portables para Windows, Linux y Colab.
from pathlib import Path

# Ejecuta comandos externos cuando una celda lo requiere.
import subprocess

# Expone el interprete Python activo.
import sys

# Instala el proyecto solo cuando la bandera esta activa.
if RUN_COLAB_INSTALL:
    # Clona la rama publica del proyecto CityLearn v3 MADRL.
    subprocess.run(['git', 'clone', '--branch', 'citylearn-v3-madrl', 'https://github.com/Mac-Tapia/CityLearn.git'], check=True)
    # Cambia el directorio activo al repositorio clonado.
    os.chdir('CityLearn')
    # Instala CityLearn localmente en modo editable.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
# Mantiene la celda segura cuando no se desea instalar.
else:
    # Informa que no se modifico el entorno actual.
    print('RUN_COLAB_INSTALL=False; usando el entorno actual.')

## Load the Current Project Dataset

El proyecto no depende del registro interno `DataSet` para cargar `citylearn_iquitos_2023_2025`. Se usa directamente el `schema.json` versionado, que es el mismo camino que deben usar Docker, AWS y otra computadora local.


In [ ]:
# Configuracion portable para cargar el dataset actual del proyecto.
from pathlib import Path
import sys

PROJECT_DATASET_NAME = 'citylearn_iquitos_2023_2025'


def find_citylearn_root(start=None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        direct_schema = candidate / 'data' / 'datasets' / PROJECT_DATASET_NAME / 'schema.json'
        nested_schema = candidate / 'CityLearn' / 'data' / 'datasets' / PROJECT_DATASET_NAME / 'schema.json'
        if direct_schema.is_file() and (candidate / 'citylearn').is_dir():
            return candidate
        if nested_schema.is_file() and (candidate / 'CityLearn' / 'citylearn').is_dir():
            return candidate / 'CityLearn'
    raise FileNotFoundError(
        f'No se encontro CityLearn/data/datasets/{PROJECT_DATASET_NAME}/schema.json desde {start}'
    )


CITYLEARN_ROOT = find_citylearn_root()
PROJECT_DATASET_ROOT = CITYLEARN_ROOT / 'data' / 'datasets' / PROJECT_DATASET_NAME
PROJECT_SCHEMA_PATH = PROJECT_DATASET_ROOT / 'schema.json'

if str(CITYLEARN_ROOT) not in sys.path:
    sys.path.insert(0, str(CITYLEARN_ROOT))

print('CITYLEARN_ROOT =', CITYLEARN_ROOT)
print('PROJECT_DATASET_NAME =', PROJECT_DATASET_NAME)
print('PROJECT_SCHEMA_PATH =', PROJECT_SCHEMA_PATH)
print('schema exists =', PROJECT_SCHEMA_PATH.is_file())

local_datasets = sorted(path.name for path in (CITYLEARN_ROOT / 'data' / 'datasets').iterdir() if path.is_dir())
dataset_available = PROJECT_DATASET_NAME in local_datasets

print('Project dataset:', PROJECT_DATASET_NAME)
print('Available in local repository:', dataset_available)
print('Local datasets:', local_datasets)


Initialize the environment using the project factory. This is the preferred path for thesis experiments because it fixes the default schema, Dec-POMDP wrapper, reward aggregation and CityLearn v3 reward function.

In [ ]:
# Importa la fabrica del entorno CityLearn v3 del proyecto.
from citylearn.v3 import describe_environment, make_citylearn_v3_project_env

# Define el eje que se desea cargar para una prueba corta.
SCENARIO = 'E1'

# Define el perfil MADRL usado por la recompensa.
MADRL_ALGORITHM = 'HAPPO'

# Define una semilla reproducible.
RANDOM_SEED = 0

# Define un horizonte corto para que el notebook cargue rapido.
TUTORIAL_EPISODE_TIME_STEPS = 24

# Crea el Dec-POMDP CityLearn v3 con 17 edificios + EV.
env = make_citylearn_v3_project_env(
    # Selecciona el eje del proyecto.
    scenario=SCENARIO,
    # Usa la semilla definida arriba.
    seed=RANDOM_SEED,
    # Acorta el horizonte solo para inspeccion.
    episode_time_steps=TUTORIAL_EPISODE_TIME_STEPS,
    # Activa el perfil reward del algoritmo seleccionado.
    madrl_algorithm=MADRL_ALGORITHM,
)

# Resume el entorno cargado.
description = describe_environment(env)

# Imprime los campos principales.
print('version_layer:', description['version_layer'])

# Imprime el simulador base.
print('simulator:', description['simulator'])

# Imprime numero de agentes.
print('num_agents:', description['num_agents'])

# Imprime dimension del estado global CTDE.
print('state_dim:', description['state_dim'])

# Imprime funcion reward activa.
print('reward_function:', description['reward_function'])

# Imprime metadatos reward del perfil MADRL.
print('reward_metadata:', description['reward_metadata'])

The dataset can also be copied to a path of choice for inspection. In this repository, the project schema already exists under `data/datasets/citylearn_iquitos_2023_2025/schema.json`, so copying is optional and local.


In [ ]:
# Controla si se copia el dataset local a una carpeta de inspeccion.
COPY_DATASET_FOR_INSPECTION = False

from pathlib import Path
import shutil

schema_filepath = PROJECT_SCHEMA_PATH

if COPY_DATASET_FOR_INSPECTION:
    target_root = Path('citylearn_dataset') / PROJECT_DATASET_NAME
    shutil.copytree(PROJECT_DATASET_ROOT, target_root, dirs_exist_ok=True)
    schema_filepath = target_root / 'schema.json'

print('Schema filepath:', schema_filepath)
print('Dataset root:', schema_filepath.parent)
print('Dataset files sample:', sorted(path.name for path in schema_filepath.parent.iterdir())[:10])


## Load an Environment Using Schema Filepath

The schema filepath can be used to initialize a CityLearn v3 Dec-POMDP environment. This approach is useful when changing datasets while preserving the v3 MADRL layer.

In [ ]:
# Importa la fabrica generica de CityLearn v3.
from citylearn.v3 import make_citylearn_v3_env

# Carga el entorno desde un schema filepath explicito.
env_from_filepath = make_citylearn_v3_env(
    # Pasa la ruta del schema local.
    schema_path=schema_filepath,
    # Usa el eje de emisiones CO2 para demostrar cambio de escenario.
    scenario='E2',
    # Usa la misma semilla reproducible.
    seed=RANDOM_SEED,
    # Mantiene horizonte corto para inspeccion.
    episode_time_steps=TUTORIAL_EPISODE_TIME_STEPS,
    # Activa el perfil reward de MASAC.
    madrl_algorithm='MASAC',
)

# Resume el entorno cargado desde filepath.
filepath_description = describe_environment(env_from_filepath)

# Imprime el escenario activo.
print('scenario:', filepath_description['scenario'])

# Imprime el numero de agentes.
print('num_agents:', filepath_description['num_agents'])

# Imprime si existen acciones EV.
print('has_ev_actions:', filepath_description['has_ev_actions'])

# Imprime metadatos reward.
print('reward_metadata:', filepath_description['reward_metadata'])

This approach is best if using a custom CityLearn v2 dataset that should be exposed through the same CityLearn v3 Dec-POMDP contract.

## Load an Environment Using Schema Dictionary Object

Alternatively, the schema can be supplied as a `dict` object. This is useful when you need to modify schema values before constructing the simulator. With CityLearn v3, the base `CityLearnEnv` is created from the dictionary and then wrapped as `CityLearnDecPOMDPEnv`.

In [ ]:
# Importa el simulador base CityLearn v2.
from citylearn.citylearn import CityLearnEnv

# Importa el wrapper Dec-POMDP del proyecto.
from citylearn.dec_pomdp import CityLearnDecPOMDPEnv

# Importa el gestor de escenarios E1/E2/E3.
from citylearn.scenario_manager import ScenarioManager

# Importa lector JSON usado por CityLearn.
from citylearn.utilities import FileHandler

# Lee el schema como diccionario Python.
schema = FileHandler.read_json(schema_filepath)

# Establece root_directory explicitamente para rutas relativas del schema.
schema['root_directory'] = str(schema_filepath.parent)

# Crea el simulador base con recompensa CityLearn v3 MADRL.
base_env = CityLearnEnv(
    # Usa el schema editado en memoria.
    schema,
    # Mantiene ejecucion descentralizada por edificio.
    central_agent=False,
    # Acorta el episodio para inspeccion.
    episode_time_steps=TUTORIAL_EPISODE_TIME_STEPS,
    # Usa semilla reproducible.
    random_seed=RANDOM_SEED,
    # Usa ejecucion offline para experimento local.
    offline=True,
    # Activa la recompensa propia CityLearn v3.
    reward_function='citylearn.reward_function.CityLearnV3MADRLRewardFunction',
    # Pasa el eje y perfil MADRL a la reward.
    reward_function_kwargs={'scenario': 'E3', 'algorithm': 'MATD3'},
)

# Crea el gestor de escenarios.
manager = ScenarioManager()

# Selecciona el eje de costos.
manager.select_scenario('E3')

# Aplica modificaciones del escenario al simulador base.
manager.apply_scenario_modifications(base_env)

# Envuelve CityLearn v2 como Dec-POMDP cooperativo.
env_from_dict = CityLearnDecPOMDPEnv(base_env, reward_aggregation='team_mean', scenario='E3')

# Resume el entorno cargado desde dict.
dict_description = describe_environment(env_from_dict)

# Imprime escenario y reward.
print('scenario:', dict_description['scenario'])

# Imprime funcion reward.
print('reward_function:', dict_description['reward_function'])

# Imprime dimensiones de accion del primer agente.
print('first_action_dim:', next(iter(dict_description['action_dims'].values())))

Some schema parameters can also be overridden by parsing them directly to the `citylearn.citylearn.CityLearnEnv` constructor. This is useful for short diagnostics, time-window tests, and controlled ablation experiments before launching HAPPO, MASAC, MATD3 or MAAC.

In [ ]:
# Lee nuevamente el schema original para evitar reutilizar mutaciones previas.
override_schema = FileHandler.read_json(schema_filepath)

# Crea un simulador base con overrides directos.
override_base_env = CityLearnEnv(
    # Usa el schema como diccionario.
    override_schema,
    # Declara la raiz del dataset de forma explicita.
    root_directory=str(schema_filepath.parent),
    # Mantiene agentes descentralizados para MADRL.
    central_agent=False,
    # Desplaza el inicio de simulacion para diagnostico.
    simulation_start_time_step=10,
    # Mantiene un horizonte corto de prueba.
    episode_time_steps=TUTORIAL_EPISODE_TIME_STEPS,
    # Usa semilla reproducible.
    random_seed=RANDOM_SEED,
    # Mantiene ejecucion offline.
    offline=True,
    # Usa la recompensa propia del proyecto.
    reward_function='citylearn.reward_function.CityLearnV3MADRLRewardFunction',
    # Usa el perfil MAAC para demostrar que el reward cambia por MADRL.
    reward_function_kwargs={'scenario': 'E1', 'algorithm': 'MAAC'},
)

# Crea un nuevo gestor de escenarios.
override_manager = ScenarioManager()

# Selecciona el eje de flexibilidad.
override_manager.select_scenario('E1')

# Aplica modificaciones E1 al simulador base.
override_manager.apply_scenario_modifications(override_base_env)

# Envuelve el simulador como Dec-POMDP.
env_with_overrides = CityLearnDecPOMDPEnv(override_base_env, reward_aggregation='team_mean', scenario='E1')

# Reinicia el entorno para obtener observaciones iniciales.
observations, infos = env_with_overrides.reset(seed=RANDOM_SEED)

# Construye acciones cero validas para todos los agentes.
zero_actions = {agent: env_with_overrides.action_space(agent).sample() * 0.0 for agent in env_with_overrides.agents}

# Ejecuta un paso de prueba del Dec-POMDP.
next_observations, rewards, terminations, truncations, step_infos = env_with_overrides.step(zero_actions)

# Imprime numero de observaciones por agente.
print('agents:', len(observations))

# Imprime forma del estado global CTDE.
print('state_shape:', env_with_overrides.state().shape)

# Imprime recompensa media del primer paso.
print('reward_mean:', sum(rewards.values()) / max(len(rewards), 1))

# Imprime si el episodio termino en el primer paso.
print('terminated_any:', any(terminations.values()))